In [23]:
# Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import optuna
from sklearn.model_selection import cross_val_score

# For showing all the the Data Frame columns
%matplotlib inline
pd.set_option('display.max_columns', None) 

In [24]:
from utils import *

In [ ]:
# importing reduced dataset
df = pd.read_csv('../data/workable_data.csv')

In [ ]:
df.head()

In [ ]:
# Select target and features
X = df.drop('failure', axis=1)
y = df['failure']

In [ ]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [ ]:
# SMOTE with sampling_strategy = 0.05 
smote = SMOTE(sampling_strategy=0.05, random_state=42) 
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
# Fit the model
rf = RandomForestClassifier(
    max_depth=10,         
    min_samples_leaf=10,   
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Feature importance

In [ ]:
sort_idx = rf.feature_importances_.argsort()
plt.barh(X.columns[sort_idx], rf.feature_importances_[sort_idx])
plt.show();

## First Drop

In [ ]:
X = X.drop(['second', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'], axis=1)

In [ ]:
corr = np.abs(X.corr())

# Set up mask for triangle representation
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(16, 16))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap
sns.heatmap(corr, mask=mask, vmax=1, square=True, 
            linewidths=.5, cbar_kws={"shrink": .5},
            annot=True, fmt='.2f',   # fmt='.2f' to round to 2 decimals
            cmap='rocket', ax=ax)

plt.title('Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [ ]:
# SMOTE with sampling_strategy = 0.05 
smote = SMOTE(sampling_strategy=0.05, random_state=42) 
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
# Fit the model
rf = RandomForestClassifier(
    max_depth=10,         
    min_samples_leaf=10,   
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

In [ ]:
results = evaluate_model(
    rf,
    X_train, X_test,
    y_train, y_test,
    'Droping_columns',
    'chronological split + SMOTE sampling_strategy=0.0.5 + threshold=0.15',
    threshold=0.15
)

In [ ]:
results

### Conclution
I chose to drop the columns: 'second', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'

# Hyperparameter Optimization with Time-Series Cross-Validation

In [34]:
df_s = pd.read_csv('../data/reduced_data.csv')

In [35]:
df_s.columns

Index(['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
       'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers',
       'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses',
       'failure', 'month', 'day', 'hour', 'minute'],
      dtype='object')

In [36]:
df_s = df_s.drop(['timestamp', 'LPS', 'Pressure_switch', 'Oil_level', 
                  'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'], axis=1).reset_index(drop=True)

In [37]:
df_s.head()

,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,DV_eletric,Towers,failure,month,day
0,-0.012286,9.327429,9.311143,-0.022857,9.328000,53.521427,0.040357,False,True,False,2,1
1,-0.013000,9.260000,9.243333,-0.023333,9.259666,53.420834,0.040000,False,True,False,2,1
2,-0.012667,9.198334,9.182000,-0.022333,9.199000,53.325000,0.040000,False,True,False,2,1
3,-0.012333,9.136666,9.120667,-0.022667,9.136666,53.200000,0.040000,False,True,False,2,1
4,-0.013000,9.075666,9.060000,-0.023000,9.075666,53.129166,0.040000,False,True,False,2,1


In [38]:
df_s.isna().sum()

TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
DV_eletric         0
Towers             0
failure            0
month              0
day                0
dtype: int64

In [39]:
# Select target and features
X = df_s.drop('failure', axis=1)
y = df_s['failure'].astype(int)

In [40]:
y.dtype

dtype('int64')

In [41]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df_s) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [42]:
# make sure test data contain failure values
print(y_test.unique())
X_test.head()

[0 1]


,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,DV_eletric,Towers,month,day
176904,-0.011667,8.917334,8.905666,-0.018000,8.920333,66.016670,0.040417,False,True,6,29
176905,-0.012000,8.750000,8.739333,-0.018333,8.753000,64.762500,0.040417,False,True,6,29
176906,-0.013000,8.628000,8.617333,-0.018667,8.630667,64.245834,0.038750,False,True,6,29
176907,-0.013000,8.527667,8.516334,-0.018333,8.529000,63.570835,0.039583,False,True,6,29
176908,-0.012667,8.439667,8.428333,-0.018667,8.441000,62.841667,0.040000,False,True,6,29


In [ ]:
# from sklearn.metrics import make_scorer, recall_score
# # Define parameter distribution for sampling
# param_dist = {
#     'n_estimators': [100, 200, 300, 500],   # number of trees
#     'max_depth': [None, 10, 20, 30],        # tree depth
#     'min_samples_split': [2, 5, 10],        # min samples to split
#     'min_samples_leaf': [1, 2, 4],          # min samples per leaf
#     'max_features': ['sqrt', 'log2', None]  # number of features per split
# }

# # Create the scorer object
# recall_class1 = make_scorer(recall_score, pos_label=True)
# # Randomized search
# random_search = RandomizedSearchCV(
#     estimator=RandomForestClassifier(
#         class_weight='balanced',
#         random_state=42,
#         n_jobs=-1),
#     param_distributions=param_dist,
#     n_iter=20,                 # number of random combinations to try
#     cv=3,                      # 3-fold cross-validation (faster than 5)
#     scoring=recall_class1,
#     n_jobs=-1,                 # use all CPU cores
#     verbose=2,
#     random_state=42
# )
# # Fit randomized search
# random_search.fit(X_train, y_train)

# # Best parameters
# print("Best parameters:", random_search.best_params_)

# # Best model
# best_rf = random_search.best_estimator_

In [ ]:
# # Evaluate on test set
# y_prob = best_rf.predict_proba(X_test)[:, 1]
# y_pred = (y_prob >= 0.15).astype(int)

# print(classification_report(y_test, y_pred, zero_division=0))

In [43]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score
from imblearn.over_sampling import SMOTE

def objective(trial):
    # --- SMOTE Param ---
    sampling_strat = trial.suggest_categorical('sampling_strategy', [0.1, 0.5, 1.0])
    
    # --- Random Forest Extended Params ---
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 5, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample']),
        'n_jobs': -1,
        'random_state': 42
    }
    
    # --- Threshold Param ---
    threshold = trial.suggest_float('threshold', 0.1, 0.5) # Keeping it low for Recall

    tscv = TimeSeriesSplit(n_splits=3)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_t, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_t, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = Pipeline([
            ('smote', SMOTE(sampling_strategy=sampling_strat, random_state=42)),
            ('rf', RandomForestClassifier(**rf_params))
        ])

        model.fit(X_t, y_t)
        
        y_prob = model.predict_proba(X_v)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)

        # F2-Score: Prioritizes Recall while respecting F1/Precision
        val_score = fbeta_score(y_v, y_pred, beta=2, pos_label=1, zero_division=0)
        scores.append(val_score)

    return np.mean(scores)

In [ ]:
# 1. Run the study with a timeout or trial limit
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) 

# 2. Extract ALL hyperparameters
best_params = study.best_params
print("Found the best configuration:", best_params)

# 3. Separate the 'Meta-Parameters' from the 'Model-Parameters'
# We pop them out of the dictionary so we can pass the rest to RandomForest
best_threshold = best_params.pop('threshold')
best_sampling_strat = best_params.pop('sampling_strategy')

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

final_model = Pipeline([
    ('smote', SMOTE(sampling_strategy=best_sampling_strat, random_state=42)),
    ('rf', RandomForestClassifier(**best_params, n_jobs=-1, random_state=42))
])
final_model.fit(X_train, y_train)

In [ ]:
y_probs = final_model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= best_threshold).astype(int)